In [ ]:
import torch
import dnnlib
import legacy
import numpy as np
import csv
import os

In [6]:
network_pkl = "ffhq.pkl"
device = torch.device("cuda")
with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to(device)

In [ ]:
affines = []

for name, module in G.synthesis.named_modules():
    if hasattr(module, "affine"):
        affines.append(module.affine)

mapping = G.mapping

In [8]:
class StyleAffineMapper(torch.nn.Module):
    def __init__(self, mapping, affines):
        super().__init__()
        self.mapping = mapping
        self.affines = torch.nn.ModuleList(affines)

    def forward(self, z, truncation=0.5):
        w = self.mapping(z, None, truncation_psi=truncation)  # shape [batch, 14, 512]

        outputs = []

        # Handle First Block
        outputs.append(self.affines[0](w[:, 0]))  # conv1
        outputs.append(self.affines[1](w[:, 1]))  # toRGB

        # Rest of the blocks
        w_idx = 2
        affine_idx = 2
        for _ in range(8):
            # conv0
            outputs.append(self.affines[affine_idx](w[:, w_idx]))
            # conv1
            outputs.append(self.affines[affine_idx + 1](w[:, w_idx + 1]))
            # toRGB (reuses the second w vector of this block)
            outputs.append(self.affines[affine_idx + 2](w[:, w_idx + 1]))

            w_idx += 2
            affine_idx += 3

        return outputs

In [9]:
class StyleSynthesisNetwork(torch.nn.Module):
    def __init__(self, synthesis):
        super().__init__()
        self.synthesis = synthesis
        
        for name, module in self.synthesis.named_modules():
            if hasattr(module, 'affine'):
                module.affine = torch.nn.Identity()

    def forward(self, precomputed_styles):

        style_idx = 0
        
        def hooked_forward(module, input):
            nonlocal style_idx
            new_input = list(input)
            new_input[1] = precomputed_styles[style_idx]
            style_idx += 1
            return tuple(new_input)

        hooks = []
        for name, module in self.synthesis.named_modules():
            if hasattr(module, 'affine'):
                hooks.append(module.register_forward_pre_hook(hooked_forward)) # pre-hook to replace the style input with precomputed styles

        try:
            dummy_ws = torch.zeros(precomputed_styles[0].shape[0],
                                   self.synthesis.num_ws, 512).to(precomputed_styles[0].device) # use a dummy tensor since styles are loaded via hooks
            img = self.synthesis(dummy_ws)
        finally:
            for h in hooks:
                h.remove()
                
        return img

In [16]:
def save_styles(styles,key,seed,folder):
    flattened_styles = []
    headers = []
    
    for idx, style in enumerate(styles, 1):
        flat = style.cpu().detach().numpy().flatten().tolist()
        for i, val in enumerate(flat):
            headers.append(f"style_vector_{idx}_dim{i}")
            flattened_styles.append(val)
    
    # Write to CSV
    with open(f'{folder}/{key}_style_{seed}.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(flattened_styles)

In [17]:
style_generator = StyleAffineMapper(mapping, affines)
style_synthesis = StyleSynthesisNetwork(G.synthesis)

folder = 'preset_styles'
os.makedirs(folder, exist_ok=True)

styles_set = []
seeds = {"male_young":[16,76,87,189,223,234,330,384,420],
         "male_middle":[12,269,270,281,282,339,364,372,386],
         "male_old":[8,191,219,315,341,374,390,401,411],
         "female_young":[6,163,92,110,119,359,369,433,485],
         "female_middle":[0,10,28,35,172,355,361,381,498],
         "female_old":[175,311,326,327,365,380,422,447,494]}

for key, seed_category in seeds.items():
    for seed in seed_category:
        z = torch.from_numpy(np.random.RandomState(seed).randn(1, G.z_dim)).to(device)
        styles = style_generator(z, truncation=0.5)
        save_styles(styles,key,seed,folder)